# Q-learning propagates future value

An agent starts at state 0 and wants to reach state 4. It may move one or two steps. Q-learning updates each action using its immediate reward plus the estimated value of the next state.

## 1. Q-learning with NumPy
Use an array as the Q-table and NumPy’s random generator for exploration.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

rng = np.random.default_rng(2)
Q = np.zeros((5, 2))
alpha, gamma, epsilon = .2, .95, .2
history = []

for episode in range(1000):
    state = 0
    while state < 4:
        action = rng.integers(2) if rng.random() < epsilon else Q[state].argmax()
        step = 1 if action == 0 else 2
        next_state = min(4, state + step)
        reward = 1.0 if next_state == 4 else -0.02
        target = reward + gamma * Q[next_state].max()
        Q[state, action] += alpha * (target - Q[state, action])
        state = next_state
    history.append(Q[0].max())

print('Learned action values by state:')
print(Q.round(3))
state, path = 0, [0]
while state < 4:
    state = min(4, state + (1 if Q[state].argmax() == 0 else 2))
    path.append(state)
print('Greedy path:', path)
plt.plot(history); plt.xlabel('episode'); plt.ylabel('estimated value at start'); plt.title('Future reward propagates back to state 0'); plt.show();

## 2. Build the same routine from scratch
Use only Python lists and `random` so every part of the Q-learning update remains visible.

In [ ]:
import random

random.seed(2)
Q_scratch = [[0.0, 0.0] for _ in range(5)]
for episode in range(1000):
    state = 0
    while state < 4:
        explore = random.random() < .2
        action = random.randrange(2) if explore else max(range(2), key=lambda a: Q_scratch[state][a])
        next_state = min(4, state + (2 if action else 1))
        reward = 1.0 if next_state == 4 else -.02
        target = reward + .95 * max(Q_scratch[next_state])
        Q_scratch[state][action] += .2 * (target - Q_scratch[state][action])
        state = next_state

for state, values in enumerate(Q_scratch):
    print(state, [round(value, 3) for value in values])

## What to notice
State 0 receives no terminal reward directly. Its value rises because its actions lead to states from which reward is reachable. That continuation value is the defining ingredient missing from a one-step decision.